# Clase 1: Fundamentos de IA Generativa
## Taller único: del ticket desordenado a una respuesta operativa

**Capacitación en Inteligencia Artificial y Automatización Aplicada a Soporte Técnico**  
**Casinos Play**

Este notebook utiliza exclusivamente un modelo local:

- **Modelo:** `LFM2.5-1.2B-Instruct`
- **Cuantización:** `Q4_K_M` de 4 bits
- **Framework:** `llama.cpp`, mediante `llama-cpp-python`
- **Entorno recomendado:** Google Colab
- **APIs externas:** ninguna

> El modelo se descarga desde Hugging Face durante la primera ejecución. Luego, la inferencia se realiza dentro de la sesión de Colab.


## Objetivo de la actividad

Transformar un ticket desordenado en una respuesta operativa mediante **tres pruebas sobre el mismo caso**:

1. Ejecutar una instrucción mínima.
2. Construir un prompt con rol, contexto, objetivo, restricciones y formato.
3. Probar la robustez del prompt incorporando información nueva.

La actividad permite observar cómo el diseño del prompt modifica la calidad, estructura y seguridad de la respuesta.

### Resultado esperado

Al finalizar, el grupo contará con:

- una comparación entre un prompt mínimo y uno estructurado;
- una lista de supuestos o errores detectados;
- un prompt reutilizable para organizar tickets;
- una primera aproximación práctica a la ingeniería de prompt.


## Dinámica presencial sugerida

- **Modalidad:** parejas o grupos de tres personas.
- **Caso:** todos trabajan con el mismo ticket.
- **Duración estimada:** 45 a 55 minutos.
- **Consigna:** modificar únicamente las celdas señaladas.

```text
Prueba inicial → prompt estructurado → cambio del caso → conclusión
```

No es necesario comprender el código de carga del modelo. El trabajo se concentra en redactar, ejecutar y evaluar prompts.


# 1. Preparación del entorno

Ejecutar las siguientes celdas en orden.

> En Google Colab, la instalación inicial puede demorar algunos minutos. No es necesario activar una GPU para este modelo. Si Colab solicita reiniciar la sesión después de instalar, reiniciarla y continuar desde la carga del modelo.


In [ ]:
# Instalar llama.cpp para Python y el cliente de Hugging Face.
# La opción --prefer-binary intenta utilizar una rueda precompilada cuando está disponible.
%pip install -q --upgrade --prefer-binary llama-cpp-python huggingface_hub


In [ ]:
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
from llama_cpp import Llama

REPO_ID = "LiquidAI/LFM2.5-1.2B-Instruct-GGUF"
CUANTIZACION = "Q4_K_M"

# Buscar automáticamente el archivo GGUF Q4_K_M del repositorio oficial.
archivos = HfApi().list_repo_files(REPO_ID)
candidatos = [
    nombre for nombre in archivos
    if nombre.lower().endswith(".gguf") and CUANTIZACION.lower() in nombre.lower()
]

if not candidatos:
    raise RuntimeError(
        f"No se encontró un archivo {CUANTIZACION} en {REPO_ID}. "
        "Revisá los archivos disponibles en el repositorio."
    )

# Si hubiera más de uno, se prioriza el que contenga 'Instruct'.
candidatos.sort(key=lambda x: ("instruct" not in x.lower(), len(x)))
NOMBRE_ARCHIVO = candidatos[0]

print("Archivo seleccionado:", NOMBRE_ARCHIVO)

ruta_modelo = hf_hub_download(
    repo_id=REPO_ID,
    filename=NOMBRE_ARCHIVO,
)

print("Modelo descargado en:", ruta_modelo)


In [ ]:
# Cargar el modelo con llama.cpp.
# n_ctx=4096 mantiene un consumo moderado para la actividad.
llm = Llama(
    model_path=ruta_modelo,
    n_ctx=4096,
    n_threads=None,
    n_gpu_layers=0,
    verbose=False,
)

print("Modelo local cargado correctamente.")


## Función de consulta

La siguiente función recibe un prompt y devuelve la respuesta del modelo.

Los parámetros se mantienen constantes durante las tres pruebas para que la comparación sea válida.


In [ ]:
def consultar_modelo(
    prompt,
    system_prompt="Sos un asistente de soporte técnico. Respondé en español.",
    max_tokens=500,
):
    respuesta = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        temperature=0.1,
        top_p=0.1,
        top_k=50,
        repeat_penalty=1.05,
        max_tokens=max_tokens,
    )
    return respuesta["choices"][0]["message"]["content"].strip()


In [ ]:
# Prueba técnica breve para comprobar que el modelo responde.
print(consultar_modelo(
    "Respondé únicamente: Entorno preparado.",
    max_tokens=20,
))


# 2. Caso de trabajo

Todos los grupos utilizarán inicialmente el mismo ticket.

## Ticket original

```text
usuario llama porque no entra
ayer funcionaba
sale error timeout
internet aparentemente anda
otro puesto parece tener el mismo problema
no sabemos desde qué hora
todavía no hay captura
pregunta cuándo va a volver
```

Antes de usar el modelo, conversar brevemente:

- ¿Qué datos están confirmados?
- ¿Qué datos son vagos o ambiguos?
- ¿Qué información falta?
- ¿Qué no debería afirmar todavía el equipo de soporte?


In [ ]:
ticket_original = """
usuario llama porque no entra
ayer funcionaba
sale error timeout
internet aparentemente anda
otro puesto parece tener el mismo problema
no sabemos desde qué hora
todavía no hay captura
pregunta cuándo va a volver
""".strip()

print(ticket_original)


# 3. Prueba 1: instrucción mínima

Ejecutar una instrucción deliberadamente poco precisa.

## Preguntas de observación

- ¿El modelo organiza la información?
- ¿Diferencia hechos de hipótesis?
- ¿Inventa causas o resultados?
- ¿Solicita información adicional?
- ¿Propone acciones seguras?
- ¿La respuesta podría utilizarse directamente?


In [ ]:
prompt_minimo = f"""
Organizá este ticket y decime qué hacer:

{ticket_original}
""".strip()

respuesta_minima = consultar_modelo(prompt_minimo)
print(respuesta_minima)


## Registro breve de la Prueba 1

Completar en grupo:

- **Algo útil de la respuesta:**
- **Una suposición o riesgo detectado:**
- **Información que el modelo debería haber pedido:**
- **Parte de la respuesta que requeriría validación humana:**


# 4. Prueba 2: prompt estructurado

Ahora se utilizarán cinco componentes básicos de prompting:

1. **Rol:** desde qué función debe responder el modelo.
2. **Contexto:** cuál es la situación.
3. **Objetivo:** qué transformación debe realizar.
4. **Restricciones:** qué debe evitar o aclarar.
5. **Formato:** cómo debe organizar la salida.

La celda contiene una versión base. Cada grupo debe leerla y puede mejorarla antes de ejecutarla.


In [ ]:
prompt_estructurado = f"""
ROL:
Actuá como asistente de soporte técnico de primer nivel.
Tu función es organizar la información, detectar datos faltantes y proponer verificaciones iniciales.
No tenés acceso real a servidores, registros, métricas ni herramientas internas.

CONTEXTO:
Recibimos las siguientes notas desordenadas de un incidente:

{ticket_original}

OBJETIVO:
Transformá las notas en un registro operativo que ayude al equipo a continuar el diagnóstico y a comunicarse con el usuario.

RESTRICCIONES:
- No inventes datos, métricas, causas ni resultados de pruebas.
- Diferenciá hechos confirmados, información ambigua e hipótesis.
- No afirmes cuándo se resolverá el incidente.
- No solicites contraseñas, credenciales ni información sensible.
- No recomiendes reinicios, cambios de configuración o acciones disruptivas sin autorización.
- Cuando no exista información suficiente, indicalo expresamente.

FORMATO DE SALIDA:
1. Resumen breve del incidente.
2. Hechos confirmados.
3. Datos ambiguos o no confirmados.
4. Información faltante.
5. Preguntas de diagnóstico.
6. Verificaciones iniciales no disruptivas.
7. Borrador breve de respuesta para el usuario.
""".strip()

respuesta_estructurada = consultar_modelo(prompt_estructurado, max_tokens=700)
print(respuesta_estructurada)


## Comparación entre las Pruebas 1 y 2

Responder únicamente estas preguntas:

1. ¿Qué cambio produjo la mejora más visible?
2. ¿La segunda respuesta separó adecuadamente hechos e hipótesis?
3. ¿Qué restricción fue más importante para un entorno de soporte?
4. ¿Qué problema todavía conserva la respuesta?

> No buscamos una respuesta perfecta. Buscamos reconocer qué aspectos puede controlar un prompt y cuáles dependen de las capacidades del modelo y de la validación humana.


# 5. Prueba 3: robustez ante información nueva

Sin cambiar el prompt estructurado, se modifica la entrada:

- El problema ahora podría afectar a diez puestos.
- No está confirmado que todos pertenezcan al mismo sector.
- Una persona afirma que el problema comenzó después de una actualización, pero todavía no se verificó.

El objetivo es comprobar si el prompt sigue funcionando cuando el caso cambia.


In [ ]:
ticket_actualizado = """
usuario llama porque no entra
ayer funcionaba
sale error timeout
internet aparentemente anda
ahora informan que podría haber diez puestos afectados
no está confirmado si todos pertenecen al mismo sector
una persona dice que comenzó después de una actualización, pero no fue verificado
no sabemos la hora exacta de inicio
todavía no hay captura ni registros revisados
preguntan cuándo va a volver
""".strip()

prompt_robustez = prompt_estructurado.replace(ticket_original, ticket_actualizado)

respuesta_robustez = consultar_modelo(prompt_robustez, max_tokens=700)
print(respuesta_robustez)


## Evaluación de robustez

Revisar si el modelo:

- mantiene el formato solicitado;
- trata los diez puestos como un dato todavía no confirmado;
- evita afirmar que la actualización fue la causa;
- solicita validar alcance, sector y horario;
- evita comprometer un tiempo de resolución;
- conserva las restricciones de seguridad.

### Conclusión del grupo

Completar:

```text
El prompt funcionó bien cuando:

El prompt debería mejorar en:

La instrucción más importante fue:

Antes de usar la respuesta en un incidente real, validaríamos:
```


# 6. Desafío final: mejorar una sola parte

Cada grupo modifica **una sola parte** del prompt estructurado:

- rol;
- contexto;
- objetivo;
- restricciones;
- formato de salida.

Luego ejecuta nuevamente el ticket actualizado y evalúa si la modificación produjo una mejora concreta.

> Cambiar una única parte permite identificar con mayor claridad qué modificación produjo el efecto observado.


In [ ]:
# Copiar aquí el prompt estructurado y modificar UNA sola parte.
prompt_final_del_grupo = prompt_robustez

respuesta_final = consultar_modelo(prompt_final_del_grupo, max_tokens=700)
print(respuesta_final)


# 7. Puesta en común

Cada grupo comparte en un máximo de dos minutos:

1. La parte del prompt que modificó.
2. El problema que intentó corregir.
3. El cambio observado en la respuesta.
4. Una limitación que el prompt no pudo resolver.

## Ideas centrales

- Un prompt claro puede mejorar estructura, foco y seguridad.
- Más texto no garantiza una mejor respuesta.
- Deben separarse hechos, hipótesis e información faltante.
- Las restricciones reducen riesgos, pero no garantizan obediencia perfecta.
- Un modelo pequeño permite experimentar localmente, pero conserva limitaciones.
- La respuesta siempre debe validarse antes de utilizarse en un caso real.


# 8. Cierre y conexión con la Clase 2

En la próxima clase se profundizará sobre:

- anatomía de un prompt;
- rol y contexto;
- objetivos y restricciones;
- formatos de salida;
- zero-shot y few-shot prompting;
- errores frecuentes;
- mejora iterativa;
- construcción de prompts reutilizables.

## Prompt reutilizable obtenido

El `prompt_estructurado` de este notebook funciona como primera versión de una plantilla para transformar notas desordenadas en un registro operativo. En la Clase 2 podrá evaluarse, simplificarse y adaptarse a los procedimientos reales de Casinos Play.


---

## Nota técnica

- El notebook utiliza solo inferencia local con `llama.cpp`.
- No solicita claves ni utiliza APIs de modelos comerciales.
- La descarga inicial del GGUF requiere conexión a Internet.
- Los parámetros de generación se mantienen fijos para hacer comparables las pruebas.
- Se utiliza un contexto de 4096 tokens para moderar el consumo durante la actividad.
- Conviene ejecutar el notebook completo antes de la clase para comprobar disponibilidad del paquete, archivo GGUF y tiempos de descarga en Colab.
